# Stage 4 G2 — Tiny RL sanity on Qwen3.6-35B-A3B (v2, strict Qwen3.5-4B replication)

**v2 corrections from v1**:
- `LR=3e-6` (Qwen3.5-4B card: `1e-6` stalled, had to bump to `3e-6` — don't repeat the mistake)
- **R2 uses raw contrastive direction at L22** (peak contrastive per S4-L0), computed empirically from 50 labeled SuperGPQA responses. This matches Qwen3.5-4B G2's methodology exactly (they used raw L13 direction from 50 labeled GSM8K responses). The previous v1 used a W_dec projection — different ablation, not what we want to report.
- `max_new=2048` uniform across pre-eval, training, mid-eval, final-eval. No artificial truncation asymmetry.
- `tqdm` + running accuracy on all eval phases.
- Raw L22 direction is cached + saved to Drive + HF so it's reproducible.

**Three-way ablation (same structure as Qwen3.5-4B G2)**:
- **R0** outcome only (SuperGPQA letter exact match)
- **R1** outcome + **SAE-sparse** per-token reward (TopK features at L23, helpful − harmful)
- **R2** outcome + **raw L22 direction** per-token reward (empirical direction from labeled data)

**Pre-registered thresholds** (identical to Qwen3.5 G2 targets):
- PASS C2: R1 ≥ R0 + 2 pp
- PASS sparse>raw: R1 − R2 ≥ 5 pp (Qwen3.5-4B saw 11 pp; allow slack for 4× fewer rollouts/step)
- FAIL: R1 ≈ R0 within 1 pp

**Target hardware**: RTX 6000 Blackwell 96 GB (or H100 80 GB, tight).

**Time budget**: ~10 h end-to-end.
- Raw-direction computation: ~50 min (50 Q × ~60 s)
- 3 ablations × ~3 h each = ~9 h

**Inputs**:
- Qwen3.6-35B-A3B base
- SAE at `caiovicentino1/Qwen3.6-35B-A3B-SAE-L23-topk-wip/sae_step_22674_92M_tokens.pt`
- Pack at `catalogs/qwen3.6-35b-a3b/reasoning_pack.json` (ρ=0.5217 validated at G1)
- SuperGPQA (m-a-p/SuperGPQA) — splits disjoint from S4-L0 [0:100] and G1 [100:250]

**Outputs**:
- `raw_direction_L22.pt` (reproducible R2 direction)
- Three LoRA adapters: `g2/R0/final`, `g2/R1/final`, `g2/R2/final`
- `s4_g2_summary.json` with verdict

## Cell 1 — Install env (fla + causal-conv1d + transformers main)

In [ ]:
import sys, subprocess, os, shutil, site
from pathlib import Path

def _pip(*args, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *args], check=check)

def _check():
    try:
        import transformers
        try:
            from transformers.models.auto.auto_mappings import CONFIG_MAPPING_NAMES
        except ImportError:
            from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
        return 'qwen3_5_moe' in CONFIG_MAPPING_NAMES, transformers.__version__
    except Exception as e:
        return False, f'import-error: {e}'

ok, ver = _check()
print(f'Current: transformers={ver}, qwen3_5_moe={ok}')

if not ok:
    print('\n→ Installing transformers @ main + peer deps')
    _pip('install', '-q', 'accelerate', 'peft', 'trl', 'datasets',
         'huggingface_hub==1.5.0', 'safetensors', 'sympy', 'einops',
         'scikit-learn', 'sentencepiece', 'tokenizers', 'protobuf', 'scipy')
    _pip('uninstall', '-y', 'transformers', check=False)
    _pip('uninstall', '-y', 'transformers', check=False)
    for p in sys.path + site.getsitepackages() + [site.getusersitepackages()]:
        tr = Path(p) / 'transformers'
        if tr.exists():
            shutil.rmtree(tr, ignore_errors=True)
    SRC_DIR = '/content/transformers_src'
    if os.path.exists(SRC_DIR):
        shutil.rmtree(SRC_DIR)
    subprocess.run(['git', 'clone', '--quiet',
                    'https://github.com/huggingface/transformers.git', SRC_DIR], check=True)
    _pip('install', '--force-reinstall', '--no-deps', '--no-cache-dir', SRC_DIR)
    _pip('install', '--no-cache-dir', 'flash-linear-attention', check=False)
    _pip('install', '--no-build-isolation', '--no-cache-dir', 'causal-conv1d', check=False)
    for mod in list(sys.modules):
        if mod.startswith('transformers') or mod.startswith('huggingface_hub'):
            del sys.modules[mod]
    ok, ver = _check()
    print(f'Post-install: transformers={ver}, qwen3_5_moe={ok}')
    if not ok:
        print('\n⚠️  Restart kernel and re-run this cell.')
        raise SystemExit

# HF auth + Drive
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('✅ HF authenticated')
except Exception as e:
    print(f'(HF auth skipped: {e})')

from google.colab import drive
drive.mount('/content/drive')

# Global imports
import json, time, random, re, gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm

DRIVE_ROOT = Path('/content/drive/MyDrive/mechreward/s4_qwen36')
G2_ROOT = DRIVE_ROOT / 'g2_v2'
G2_ROOT.mkdir(parents=True, exist_ok=True)
print(f'\nG2 v2 artifacts: {G2_ROOT}')

## Cell 2 — CFG (LR=3e-6 per Qwen3.5-4B lesson)

In [ ]:
CFG = dict(
    model_id='Qwen/Qwen3.6-35B-A3B',
    sae_repo='caiovicentino1/Qwen3.6-35B-A3B-SAE-L23-topk-wip',
    sae_file='sae_step_22674_92M_tokens.pt',
    sae_layer=23,
    peak_contrastive_layer=22,  # from S4-L0, this is where R2 direction is computed
    pack_repo='caiovicentino/mechreward',
    pack_path='catalogs/qwen3.6-35b-a3b/reasoning_pack.json',

    # Raw-direction computation
    raw_dir_n=50,                    # 50 labeled SuperGPQA responses (Qwen3.5-4B used 50 GSM8K)
    raw_dir_indices='[100:150]',     # same as G1 discovery set — already known balanced 22/28

    # Training (compact for 35B MoE + thinking mode)
    max_steps=30,
    batch_questions=2,
    rollouts_per_q=2,                # 4 rollouts/step total (Qwen3.5-4B used 16)
    micro_batch_logprobs=1,
    max_prompt_len=512,
    max_gen_len=2048,                # thinking mode needs ~2700 avg; truncate ~30% (accept)
    lr=3e-6,                         # CRITICAL: Qwen3.5-4B card showed 1e-6 stalls
    lora_r=32,
    lora_alpha=64,
    lora_dropout=0.0,
    lora_target=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],

    # Rewards (same as Qwen3.5-4B G2)
    lam=0.1,
    kl_beta=0.05,
    per_token_reward=True,

    # Eval — keep uniform max_new so R0/R1/R2 are comparable
    do_pre_eval=False,                # skip (we use G1 baseline 43% as reference)
    quick_eval_every=15,              # one peek mid-run
    quick_eval_n=30,
    final_eval_n=50,                  # reduced from 100 to fit time budget
    eval_max_new=2048,                # SAME as training to avoid artifacts

    # Data (disjoint from S4-L0 [0:100] and G1 [100:250])
    difficulty_filter=['easy', 'middle'],
    train_split='[350:550]',
    eval_split='[550:650]',
    seed=42,

    # Output
    ckpt_dir=str(G2_ROOT),
    hf_repo='caiovicentino1/Qwen3.6-35B-A3B-SAE-L23-topk-wip',
)
print(json.dumps({k: v for k, v in CFG.items()}, indent=2))

## Cell 3 — Load Qwen3.6-35B-A3B

In [ ]:
import transformers
try:
    from transformers.models.auto.auto_mappings import CONFIG_MAPPING_NAMES
except ImportError:
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
assert 'qwen3_5_moe' in CONFIG_MAPPING_NAMES, 'Restart kernel, re-run Cell 1.'

from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, PeftModel

tok = AutoTokenizer.from_pretrained(CFG['model_id'], trust_remote_code=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

base_model = AutoModelForImageTextToText.from_pretrained(
    CFG['model_id'], dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True,
)
for n, p in base_model.named_parameters():
    if 'visual' in n.lower() or 'vision' in n.lower():
        p.requires_grad = False

print(f'Base model loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')
print(f'Base type: {type(base_model).__name__}  (expect Qwen3_5MoeForConditionalGeneration)')

## Cell 4 — Load SAE + feature pack (for R1 sparse reward)

In [ ]:
from huggingface_hub import hf_hub_download

sae_ckpt_path = hf_hub_download(repo_id=CFG['sae_repo'], filename=CFG['sae_file'])
print(f'SAE ckpt: {sae_ckpt_path}')

sae_state = torch.load(sae_ckpt_path, map_location='cuda', weights_only=False)
if hasattr(sae_state, 'state_dict'):
    sae_state = sae_state.state_dict()
W_enc = sae_state['W_enc'].to('cuda', torch.float32)
W_dec = sae_state['W_dec'].to('cuda', torch.float32)
if W_enc.shape[0] > W_enc.shape[1]: W_enc = W_enc.t().contiguous()
if W_dec.shape[0] < W_dec.shape[1]: W_dec = W_dec.t().contiguous()
d_model, d_sae = W_enc.shape
b_enc = sae_state.get('b_enc', torch.zeros(d_sae, device='cuda')).to('cuda', torch.float32)
b_dec = sae_state.get('b_dec', torch.zeros(d_model, device='cuda')).to('cuda', torch.float32)
K = int(sae_state.get('k', 128))
print(f'SAE: d_model={d_model}, d_sae={d_sae}, k={K}')

# Feature pack from github (auto-clone if needed)
if not Path('/content/mechreward').exists():
    subprocess.run(['git', 'clone', '-q', f'https://github.com/{CFG["pack_repo"]}.git',
                    '/content/mechreward'], check=True)
with open(f'/content/mechreward/{CFG["pack_path"]}') as f:
    pack = json.load(f)
helpful_ids = [x['feature_id'] for x in pack['helpful_features']]
harmful_ids = [x['feature_id'] for x in pack['harmful_features']]
print(f'Pack: {len(helpful_ids)} helpful + {len(harmful_ids)} harmful (ρ=0.5217 G1 validated)')

ALL_FEATS = torch.tensor(helpful_ids + harmful_ids, dtype=torch.long, device='cuda')
FEAT_WEIGHTS = torch.tensor([1.0]*len(helpful_ids) + [-1.0]*len(harmful_ids),
                            dtype=torch.float32, device='cuda')
W_enc_sel = W_enc[:, ALL_FEATS].contiguous()
b_enc_sel = b_enc[ALL_FEATS].contiguous()
print(f'W_enc_sel shape: {tuple(W_enc_sel.shape)} (will use for R1 sparse encode)')

## Cell 5 — Dataset SuperGPQA (train + eval + raw-direction set)

In [ ]:
from datasets import load_dataset

raw = load_dataset('m-a-p/SuperGPQA', split='train')
filtered = raw.filter(lambda ex: ex['difficulty'] in CFG['difficulty_filter'])
shuffled = filtered.shuffle(seed=CFG['seed'])

# Splits (all disjoint):
# [0:100]     = S4-L0 probe set
# [100:150]   = G1 discovery / raw-direction set (50 Q)
# [150:250]   = G1 validation set (100 Q)
# [350:550]   = G2 train set (200 Q)
# [550:650]   = G2 final eval (use first 50 for final_eval_n, first 30 for quick_eval)

SGPQA_RAW_DIR = shuffled.select(range(100, 100 + CFG['raw_dir_n']))
SGPQA_TRAIN = shuffled.select(range(350, 550))
SGPQA_EVAL = shuffled.select(range(550, 550 + CFG['final_eval_n']))
SGPQA_QUICK = shuffled.select(range(550, 550 + CFG['quick_eval_n']))

print(f'Raw-direction set: {len(SGPQA_RAW_DIR)}')
print(f'Train: {len(SGPQA_TRAIN)}')
print(f'Quick eval: {len(SGPQA_QUICK)}  Final eval: {len(SGPQA_EVAL)}')
print(f'\nEval set sample:')
ex = SGPQA_EVAL[0]
print(f'  discipline: {ex["discipline"]}/{ex["field"]}  difficulty: {ex["difficulty"]}')
print(f'  Q: {ex["question"][:150]}')
print(f'  gold: {ex["answer_letter"]}  (#opts: {len(ex["options"])})')

## Cell 6 — Prompt + verifier (MCQ, thinking mode same as G1)

In [ ]:
def format_prompt(ex):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(ex['options']))
    msgs = [{
        'role': 'user',
        'content': (
            f"Answer the following multiple-choice question. Reason step by step, "
            f"then put the letter of the correct answer in \\boxed{{}}.\n\n"
            f"Question: {ex['question']}\n\n"
            f"Options:\n{choices}"
        )
    }]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

BOXED_RE = re.compile(r'\\boxed\{\s*([A-J])\s*\}')
LETTER_AT_END_RE = re.compile(r'[Aa]nswer[:\s]+([A-J])\b')

def extract_letter(completion, n_opts):
    m = list(BOXED_RE.finditer(completion))
    if m:
        letter = m[-1].group(1).upper()
        if ord(letter) - ord('A') < n_opts: return letter
    m2 = list(LETTER_AT_END_RE.finditer(completion))
    if m2:
        letter = m2[-1].group(1).upper()
        if ord(letter) - ord('A') < n_opts: return letter
    tail = completion[-200:]
    cands = re.findall(r'\b([A-J])\b', tail)
    if cands:
        letter = cands[-1]
        if ord(letter) - ord('A') < n_opts: return letter
    return None

def verify(ex, completion):
    pred = extract_letter(completion, len(ex['options']))
    return pred is not None and pred == ex['answer_letter']

## Cell 7 — Layer hooks + helpers

We capture hidden states at **both** L22 (for R2 raw direction) and L23 (for R1 SAE-sparse) in a single forward pass.

In [ ]:
def get_layer_module(m, idx):
    candidates = [m]
    if hasattr(m, 'base_model') and m.base_model is not m:
        candidates.append(m.base_model.model if hasattr(m.base_model, 'model') else m.base_model)
    if hasattr(m, 'model'):
        candidates.append(m.model)
    paths = [
        ('model','language_model','layers'), ('language_model','layers'),
        ('model','layers'), ('layers',),
    ]
    for start in candidates:
        for path in paths:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except (IndexError, TypeError): continue
    raise RuntimeError(f'Could not locate layer {idx}')

class DualCapture:
    """Capture hidden at two layers simultaneously (L22 for R2, L23 for R1)."""
    def __init__(self):
        self.h = {}
    def make_hook(self, layer_idx):
        def hook(module, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            self.h[layer_idx] = h.detach()
        return hook

def register_dual_capture(m, layer_indices):
    cap = DualCapture()
    handles = []
    for li in layer_indices:
        handles.append(get_layer_module(m, li).register_forward_hook(cap.make_hook(li)))
    return cap, handles

# Sanity
_l22 = get_layer_module(base_model, CFG['peak_contrastive_layer'])
_l23 = get_layer_module(base_model, CFG['sae_layer'])
print(f'L{CFG["peak_contrastive_layer"]} (R2 direction layer): {type(_l22).__name__}')
print(f'L{CFG["sae_layer"]} (R1 SAE layer): {type(_l23).__name__}')

## Cell 8 — Compute raw L22 contrastive direction (for R2)

This is the key fix from v1. We compute the direction the **same way Qwen3.5-4B did**: from labeled data (50 responses), at the peak contrastive layer (L22 here, was L13 in Qwen3.5).

Direction = `mean(h_L22 | correct) − mean(h_L22 | wrong)` / norm.

Cached to Drive so it's reproducible and doesn't need recomputation on re-runs.

In [ ]:
raw_dir_cache = G2_ROOT / 'raw_direction_L22.pt'

if raw_dir_cache.exists():
    cache = torch.load(raw_dir_cache, map_location='cuda')
    raw_direction_L22 = cache['direction']
    print(f'✅ Loaded cached raw_direction_L22 from {raw_dir_cache}')
    print(f'   Computed on {cache["n_correct"]} correct + {cache["n_wrong"]} wrong responses')
else:
    print(f'Computing raw_direction_L22 from {CFG["raw_dir_n"]} labeled SuperGPQA responses...')
    print(f'(this takes ~50 min; cached to {raw_dir_cache})')
    
    base_model.eval()
    base_model.config.use_cache = True
    
    cap, handles = register_dual_capture(base_model, [CFG['peak_contrastive_layer']])
    h_correct_sum = torch.zeros(d_model, device='cuda', dtype=torch.float32)
    h_wrong_sum = torch.zeros(d_model, device='cuda', dtype=torch.float32)
    n_correct = 0
    n_wrong = 0
    
    try:
        for ex in tqdm(SGPQA_RAW_DIR, desc='raw_dir'):
            prompt = format_prompt(ex)
            enc = tok(prompt, return_tensors='pt', truncation=True,
                      max_length=CFG['max_prompt_len']).to('cuda')
            P = enc.input_ids.shape[1]
            with torch.no_grad():
                gen = base_model.generate(
                    **enc, max_new_tokens=CFG['max_gen_len'],
                    do_sample=False, pad_token_id=tok.pad_token_id, use_cache=True,
                )
            # Second forward to capture L22 hidden
            cap.h = {}
            attn = (gen != tok.pad_token_id).long()
            with torch.no_grad():
                base_model(input_ids=gen, attention_mask=attn, use_cache=False)
            hidden = cap.h[CFG['peak_contrastive_layer']][0]  # [T, d_model]
            resp_mask = attn[0].clone()
            resp_mask[:P] = 0
            n_resp = int(resp_mask.sum().item())
            if n_resp == 0: continue
            pooled = (hidden[resp_mask.bool()].float()).mean(dim=0)  # [d_model]
            
            completion = tok.decode(gen[0][P:], skip_special_tokens=True)
            is_correct = verify(ex, completion)
            if is_correct:
                h_correct_sum += pooled
                n_correct += 1
            else:
                h_wrong_sum += pooled
                n_wrong += 1
    finally:
        for h in handles: h.remove()
    
    print(f'\nCorrect: {n_correct}, Wrong: {n_wrong}')
    assert n_correct >= 3 and n_wrong >= 3, 'Not enough samples in either group'
    
    mean_c = h_correct_sum / n_correct
    mean_w = h_wrong_sum / n_wrong
    diff = mean_c - mean_w
    raw_direction_L22 = diff / (diff.norm() + 1e-8)
    
    torch.save({
        'direction': raw_direction_L22,
        'mean_correct': mean_c,
        'mean_wrong': mean_w,
        'n_correct': n_correct,
        'n_wrong': n_wrong,
        'layer': CFG['peak_contrastive_layer'],
        'dataset': 'SuperGPQA',
        'indices': CFG['raw_dir_indices'],
    }, raw_dir_cache)
    print(f'✅ Saved raw_direction_L22 to {raw_dir_cache}')
    print(f'   direction norm: {raw_direction_L22.norm().item():.6f} (should be 1.0)')
    print(f'   diff.norm() raw: {diff.norm().item():.3f}')

# Upload to HF for reproducibility
try:
    from huggingface_hub import HfApi
    HfApi().upload_file(
        path_or_fileobj=str(raw_dir_cache),
        path_in_repo='g2_v2/raw_direction_L22.pt',
        repo_id=CFG['hf_repo'], repo_type='model',
    )
    print('✅ Uploaded raw_direction_L22 to HF')
except Exception as e:
    print(f'HF upload skipped: {e}')

## Cell 9 — Reward functions (R0, R1 sparse, R2 raw L22)

In [ ]:
def mech_reward_sparse_per_token(cap, attn_mask):
    """R1: SAE-sparse features at L23.
    cap: DualCapture object with .h populated.
    Returns [B, T] fp32."""
    hidden = cap.h[CFG['sae_layer']]
    h = hidden.to(torch.float32) - b_dec
    pre = torch.einsum('btd,df->btf', h, W_enc_sel) + b_enc_sel  # [B, T, 20]
    acts = F.relu(pre)
    r = (acts * FEAT_WEIGHTS).sum(dim=-1) * attn_mask.to(acts.dtype)
    return r

def mech_reward_raw_per_token(cap, attn_mask):
    """R2: raw direction projection at L22 (peak contrastive layer).
    Returns [B, T] fp32."""
    hidden = cap.h[CFG['peak_contrastive_layer']]
    h = hidden.to(torch.float32)  # NO b_dec subtraction — raw direction is in raw residual space
    r = torch.einsum('btd,d->bt', h, raw_direction_L22) * attn_mask.to(torch.float32)
    return r

def mech_reward_zero_per_token(cap, attn_mask):
    """R0: no mech reward."""
    return torch.zeros_like(attn_mask, dtype=torch.float32)

REWARD_FN = {
    'R0': mech_reward_zero_per_token,
    'R1': mech_reward_sparse_per_token,
    'R2': mech_reward_raw_per_token,
}
LAYERS_NEEDED = {
    'R0': [CFG['sae_layer']],  # not used but register anyway for consistency
    'R1': [CFG['sae_layer']],
    'R2': [CFG['peak_contrastive_layer']],
}
print('Reward functions ready: R0 (zero), R1 (sparse L23), R2 (raw L22)')

## Cell 10 — Eval harness (tqdm + running acc)

In [ ]:
@torch.no_grad()
def generate_one(model, prompt, max_new=None, temperature=0.0):
    max_new = max_new or CFG['eval_max_new']
    enc = tok(prompt, return_tensors='pt', truncation=True,
              max_length=CFG['max_prompt_len']).to('cuda')
    gen = model.generate(
        **enc, max_new_tokens=max_new,
        do_sample=temperature > 0, temperature=max(temperature, 1e-5),
        top_p=1.0, pad_token_id=tok.pad_token_id, use_cache=True,
    )
    return tok.decode(gen[0][enc.input_ids.shape[1]:], skip_special_tokens=True)

def eval_sgpqa(model, dataset, desc='eval'):
    model.eval()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    correct = 0
    for i, ex in enumerate(tqdm(dataset, desc=desc, leave=False)):
        prompt = format_prompt(ex)
        out = generate_one(model, prompt, max_new=CFG['eval_max_new'])
        if verify(ex, out):
            correct += 1
        if (i + 1) % 10 == 0:
            tqdm.write(f'  [{desc}] {i+1}/{len(dataset)}: running acc={correct/(i+1)*100:.1f}%')
    model.train()
    return correct / len(dataset)

print('Eval harness ready (tqdm + running acc)')

## Cell 11 — GRPO step with dual-layer capture

In [ ]:
def compute_logprobs(m, input_ids, attn_mask, response_mask, micro=None):
    micro = micro or CFG['micro_batch_logprobs']
    B = input_ids.shape[0]
    outs = []
    for i in range(0, B, micro):
        ids_c = input_ids[i:i+micro]; attn_c = attn_mask[i:i+micro]
        resp_c = response_mask[i:i+micro]
        out = m(input_ids=ids_c, attention_mask=attn_c, use_cache=False)
        logits = out.logits[:, :-1]; targets = ids_c[:, 1:]
        logp = F.log_softmax(logits, dim=-1)
        tok_logp = logp.gather(-1, targets.unsqueeze(-1)).squeeze(-1).float()
        outs.append(tok_logp * resp_c[:, 1:].float())
        del out, logits, logp, tok_logp
    return torch.cat(outs, dim=0)

def grpo_step(model, batch, step, mode):
    reward_fn = REWARD_FN[mode]
    layers_to_capture = list(set([CFG['sae_layer'], CFG['peak_contrastive_layer']]))
    lam = 0.0 if mode == 'R0' else CFG['lam']
    kl_c = CFG['kl_beta']

    prompts = [format_prompt(ex) for ex in batch]
    all_ids, all_attn, all_resp, all_outcome, all_mech_tok = [], [], [], [], []

    # ROLLOUT
    model.eval()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    cap, handles = register_dual_capture(model, layers_to_capture)
    try:
        for pi, prompt in enumerate(prompts):
            enc = tok(prompt, return_tensors='pt', truncation=True,
                      max_length=CFG['max_prompt_len']).to('cuda')
            P = enc.input_ids.shape[1]
            with torch.no_grad():
                gen = model.generate(
                    **enc, max_new_tokens=CFG['max_gen_len'],
                    do_sample=True, temperature=0.9, top_p=0.95,
                    num_return_sequences=CFG['rollouts_per_q'],
                    pad_token_id=tok.pad_token_id, use_cache=True,
                )
            attn = (gen != tok.pad_token_id).long().to('cuda')
            cap.h = {}
            with torch.no_grad():
                model(input_ids=gen, attention_mask=attn, use_cache=False)
            resp_mask = torch.zeros_like(attn)
            resp_mask[:, P:] = attn[:, P:]
            mech_tok = reward_fn(cap, resp_mask)
            completions = tok.batch_decode(gen[:, P:], skip_special_tokens=True)
            outcomes = torch.tensor([
                1.0 if verify(batch[pi], c) else 0.0 for c in completions
            ], device='cuda')
            all_ids.append(gen); all_attn.append(attn); all_resp.append(resp_mask)
            all_outcome.append(outcomes); all_mech_tok.append(mech_tok.detach())
            # Clear hidden states to free memory
            cap.h = {}
    finally:
        for h in handles: h.remove()
    torch.cuda.empty_cache()

    # Pad + concat
    maxT = max(x.shape[1] for x in all_ids)
    def pad_right(x, val=0):
        pad = maxT - x.shape[1]
        return F.pad(x, (0, pad), value=val) if pad > 0 else x
    ids = torch.cat([pad_right(x, tok.pad_token_id) for x in all_ids], dim=0)
    attn = torch.cat([pad_right(x) for x in all_attn], dim=0)
    resp = torch.cat([pad_right(x) for x in all_resp], dim=0)
    mech_tok = torch.cat([pad_right(x) for x in all_mech_tok], dim=0)
    outcomes = torch.cat(all_outcome, dim=0)

    lens = resp.sum(dim=-1).clamp(min=1).float()
    mech_traj = mech_tok.sum(dim=-1) / lens
    traj_r = outcomes + lam * mech_traj
    N = CFG['rollouts_per_q']
    traj_r = traj_r.view(-1, N)
    adv = (traj_r - traj_r.mean(dim=-1, keepdim=True)) / (traj_r.std(dim=-1, keepdim=True) + 1e-8)
    adv = adv.view(-1)
    adv_tok = adv.unsqueeze(-1) * resp[:, 1:].float()
    if CFG['per_token_reward'] and mode != 'R0':
        mt = mech_tok[:, 1:].view(-1, N, maxT - 1)
        mt = (mt - mt.mean(dim=(1,2), keepdim=True)) / (mt.std(dim=(1,2), keepdim=True) + 1e-8)
        adv_tok = adv_tok + lam * mt.view(-1, maxT - 1) * resp[:, 1:].float()

    # TRAIN
    model.train()
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    try: model.enable_input_require_grads()
    except Exception: pass

    pol_logp = compute_logprobs(model, ids, attn, resp)
    with torch.no_grad(), model.disable_adapter():
        ref_logp = compute_logprobs(model, ids, attn, resp)
    kl = pol_logp - ref_logp
    denom = resp[:, 1:].sum().clamp(min=1).float()
    loss = -(pol_logp * adv_tok).sum() / denom
    loss = loss + kl_c * (kl ** 2).sum() / denom

    return loss, dict(
        mode=mode, lam=lam,
        mean_outcome=outcomes.mean().item(),
        mean_mech=mech_traj.mean().item(),
        mean_kl=(kl.sum() / denom).item(),
    )

print('grpo_step ready')

## Cell 12 — run_ablation (fresh LoRA per ablation, clean unload)

In [ ]:
from torch.optim import AdamW

def run_ablation(mode):
    global base_model
    print(f'\n{"="*70}\n=== Running ablation: {mode} (LR={CFG["lr"]:.0e}) ===\n{"="*70}\n')

    # Defensive: strip any stale adapter
    while isinstance(base_model, PeftModel) or hasattr(base_model, 'peft_config'):
        try:
            base_model = base_model.unload()
            print(f'Unloaded stale adapter')
        except AttributeError:
            if hasattr(base_model, 'peft_config'):
                del base_model.peft_config
            break
    gc.collect(); torch.cuda.empty_cache()

    ablation_dir = G2_ROOT / mode
    ablation_dir.mkdir(parents=True, exist_ok=True)

    lora_cfg = LoraConfig(
        r=CFG['lora_r'], lora_alpha=CFG['lora_alpha'], lora_dropout=CFG['lora_dropout'],
        target_modules=CFG['lora_target'], task_type='CAUSAL_LM',
    )
    model = get_peft_model(base_model, lora_cfg)
    model.print_trainable_parameters()
    optim = AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG['lr'])

    t0 = time.time()
    history = []
    if CFG['do_pre_eval']:
        pre_acc = eval_sgpqa(model, SGPQA_QUICK, desc=f'{mode}_pre')
        print(f'[{mode}] pre-train eval (30Q): {pre_acc*100:.1f}%')
        history.append({'step': 0, 'mode': mode, 'quick_eval': pre_acc, 'elapsed_min': (time.time()-t0)/60})

    random.seed(CFG['seed'])
    indices = list(range(len(SGPQA_TRAIN)))
    random.shuffle(indices)

    for step in range(CFG['max_steps']):
        off = (step * CFG['batch_questions']) % len(SGPQA_TRAIN)
        batch_idx = [indices[(off + i) % len(SGPQA_TRAIN)] for i in range(CFG['batch_questions'])]
        batch = [SGPQA_TRAIN[i] for i in batch_idx]

        t_step = time.time()
        loss, log = grpo_step(model, batch, step + 1, mode)
        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optim.step()
        dt = time.time() - t_step

        log.update(step=step+1, loss=loss.item(), step_sec=dt, elapsed_min=(time.time()-t0)/60)
        history.append(log)

        # Every step — visible progress (don't rely on %2)
        print(f"[{mode} {step+1:2d}/{CFG['max_steps']}] loss={log['loss']:.4f} "
              f"outcome={log['mean_outcome']:.2f} mech={log['mean_mech']:.3f} "
              f"kl={log['mean_kl']:.4f} {dt:.0f}s")

        if (step + 1) == CFG['quick_eval_every']:
            mid_acc = eval_sgpqa(model, SGPQA_QUICK, desc=f'{mode}_step{step+1}')
            print(f'[{mode}] mid-eval @ step {step+1} (30Q): {mid_acc*100:.1f}%')
            history.append({'step': step+1, 'mode': mode, 'quick_eval': mid_acc,
                            'elapsed_min': (time.time()-t0)/60})
            model.save_pretrained(str(ablation_dir / f'step_{step+1}'))

    # Final full eval
    print(f'\n[{mode}] final eval ({CFG["final_eval_n"]}Q)...')
    final_acc = eval_sgpqa(model, SGPQA_EVAL, desc=f'{mode}_final')
    print(f'[{mode}] FINAL: {final_acc*100:.2f}%')

    model.save_pretrained(str(ablation_dir / 'final'))
    with open(ablation_dir / 'history.json', 'w') as f:
        json.dump(history, f, indent=2, default=float)
    with open(ablation_dir / 'final.json', 'w') as f:
        json.dump({'mode': mode, 'final_eval': final_acc,
                   'elapsed_min': (time.time()-t0)/60}, f, indent=2)

    del model, optim
    gc.collect(); torch.cuda.empty_cache()
    print(f'VRAM after unload: {torch.cuda.memory_allocated()/1e9:.1f} GB')

    return {'mode': mode, 'final': final_acc,
            'history': history, 'elapsed_min': (time.time()-t0)/60}

print('run_ablation ready')

## Cell 13 — Run 3 ablations sequentially

In [ ]:
results = {}
total_start = time.time()

for mode in ['R0', 'R1', 'R2']:
    results[mode] = run_ablation(mode)
    # Save intermediate state so we don't lose results if something crashes
    with open(G2_ROOT / f'intermediate_after_{mode}.json', 'w') as f:
        json.dump({k: {'final': v['final'], 'elapsed_min': v['elapsed_min']}
                   for k, v in results.items()}, f, indent=2)

elapsed_total_h = (time.time() - total_start) / 3600
print(f'\n{"="*70}')
print(f'ALL 3 ABLATIONS DONE ({elapsed_total_h:.1f} h total)')
print(f'{"="*70}')
for mode in ['R0', 'R1', 'R2']:
    r = results[mode]
    print(f'{mode}: final={r["final"]*100:.2f}%  ({r["elapsed_min"]:.0f}min)')

## Cell 14 — Verdict + save + HF upload

In [ ]:
r0 = results['R0']['final']; r1 = results['R1']['final']; r2 = results['R2']['final']
r1_vs_r0 = (r1 - r0) * 100
r1_vs_r2 = (r1 - r2) * 100
r2_vs_r0 = (r2 - r0) * 100

print(f'{"="*70}')
print(f'STAGE GATE 2 v2 — VERDICT (LR={CFG["lr"]:.0e}, raw L22 direction for R2)')
print(f'{"="*70}')
print(f'R0 (outcome only):                      {r0*100:.2f}%')
print(f'R1 (outcome + SAE sparse L23, λ=0.1):   {r1*100:.2f}%  (Δ R0: {r1_vs_r0:+.1f}pp)')
print(f'R2 (outcome + raw L22 direction, λ=0.1): {r2*100:.2f}%  (Δ R0: {r2_vs_r0:+.1f}pp)')
print()
print(f'R1 - R2 gap (sparse vs raw):            {r1_vs_r2:+.1f}pp')
print(f'Qwen3.5-4B G2 reference: R1-R2 = +11pp, R1-R0 = +2pp')
print()

pass_c2 = r1_vs_r0 >= 2.0
pass_sparse_beats_raw = r1_vs_r2 >= 5.0

print(f'PASS C2 (R1 ≥ R0 + 2pp):     {"✅ YES" if pass_c2 else "❌ NO"}  ({r1_vs_r0:+.1f}pp)')
print(f'PASS sparse>raw (≥5pp gap):  {"✅ YES" if pass_sparse_beats_raw else "❌ NO"}  ({r1_vs_r2:+.1f}pp)')

if pass_c2 and pass_sparse_beats_raw:
    verdict = 'PASS-EXCELLENT: G2 replicates Qwen3.5-4B pattern on triple-hybrid MoE. Go S4-G3.'
elif pass_c2:
    verdict = 'PASS-PARTIAL: C2 met but sparse-vs-raw gap < 5pp. Go G3; note smaller gap in paper.'
elif r1_vs_r0 > 0 and r2_vs_r0 < 0:
    verdict = 'MARGINAL: direction correct but magnitudes small. Likely noise; consider extending to 50 steps.'
else:
    verdict = 'FAIL: investigate. Possible: feature pack insufficient, LR wrong, SAE undertrained.'

print(f'\nVerdict: {verdict}')

summary = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'model': CFG['model_id'], 'sae_ckpt': CFG['sae_file'],
    'r2_layer': CFG['peak_contrastive_layer'], 'r1_layer': CFG['sae_layer'],
    'config': {k: v for k, v in CFG.items() if k not in ('ckpt_dir',)},
    'results': {
        'R0_final': r0, 'R1_final': r1, 'R2_final': r2,
        'R1_vs_R0_pp': r1_vs_r0, 'R1_vs_R2_pp': r1_vs_r2, 'R2_vs_R0_pp': r2_vs_r0,
    },
    'verdict': verdict,
    'pass_c2': pass_c2, 'pass_sparse_beats_raw': pass_sparse_beats_raw,
    'elapsed_h': elapsed_total_h,
}
with open(G2_ROOT / 's4_g2_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)
print(f'\n✅ Saved: {G2_ROOT}/s4_g2_summary.json')

try:
    from huggingface_hub import HfApi
    HfApi().upload_file(
        path_or_fileobj=str(G2_ROOT / 's4_g2_summary.json'),
        path_in_repo='g2_v2/s4_g2_summary.json',
        repo_id=CFG['hf_repo'], repo_type='model',
    )
    print('✅ Uploaded summary to HF')
except Exception as e:
    print(f'HF upload skipped: {e}')